# Car Semantic Search - Sentence Transformer

## 1. Cài thư viện

In [7]:
%pip install -q sentence-transformers pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import và load model
Import và load model lần chạy đầu tiên, các lần chạy sau không cần import lại.

In [8]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Đã load model:", MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2651.51it/s]


Đã load model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 3. Dữ liệu xe mẫu
Tập dataset mẫu: mỗi xe có một mô tả ngắn bằng tiếng Việt.

In [9]:
cars = [
    {"name": "Toyota Vios", "description": "Sedan 5 chỗ nhỏ gọn, tiết kiệm nhiên liệu, phù hợp đi lại trong thành phố."},
    {"name": "Mazda 3", "description": "Sedan 5 chỗ thiết kế thể thao, nội thất đẹp, phù hợp người trẻ."},
    {"name": "Hyundai Creta", "description": "SUV đô thị 5 chỗ, gầm cao, tiện nghi, phù hợp gia đình nhỏ."},
    {"name": "Toyota Innova", "description": "MPV 7 chỗ rộng rãi, phù hợp gia đình đông người và đi du lịch."},
    {"name": "Honda City", "description": "Sedan 5 chỗ tiết kiệm xăng, dễ lái, thích hợp di chuyển hằng ngày."},
    {"name": "Ford Everest", "description": "SUV 7 chỗ gầm cao, mạnh mẽ, phù hợp đi đường dài và gia đình."},
    {"name": "Mazda CX-5", "description": "SUV 5 chỗ cao cấp, thiết kế đẹp, phù hợp gia đình và đi chơi cuối tuần."},
]

df = pd.DataFrame(cars)
display(df)

,name,description
0,Toyota Vios,"Sedan 5 chỗ nhỏ gọn, tiết kiệm nhiên liệu, phù..."
1,Mazda 3,"Sedan 5 chỗ thiết kế thể thao, nội thất đẹp, p..."
2,Hyundai Creta,"SUV đô thị 5 chỗ, gầm cao, tiện nghi, phù hợp ..."
3,Toyota Innova,"MPV 7 chỗ rộng rãi, phù hợp gia đình đông ngườ..."
4,Honda City,"Sedan 5 chỗ tiết kiệm xăng, dễ lái, thích hợp ..."
5,Ford Everest,"SUV 7 chỗ gầm cao, mạnh mẽ, phù hợp đi đường d..."
6,Mazda CX-5,"SUV 5 chỗ cao cấp, thiết kế đẹp, phù hợp gia đ..."


## 4. Chuyển mô tả xe thành vector
Mỗi mô tả được mã hóa thành một vector có số chiều cố định. Chỉ lấy 8 phần tử đầu để dễ quan sát.

In [10]:
car_texts = [f"{car['name']}. {car['description']}" for car in cars]
car_embeddings = model.encode(car_texts, convert_to_numpy=True, normalize_embeddings=True)
print("Kích thước ma trận embedding:", car_embeddings.shape)
print("8 giá trị đầu của vector xe đầu tiên:", np.round(car_embeddings[0][:8], 4))

Kích thước ma trận embedding: (7, 384)
8 giá trị đầu của vector xe đầu tiên: [ 0.0757  0.0623  0.0273  0.0262 -0.0026  0.0559  0.0134  0.0129]


## 5. Nhập câu tìm kiếm và tính Cosine Similarity
Vì các vector đã được chuẩn hóa về độ dài 1, tích vô hướng bằng Cosine Similarity. 
Điểm cao hơn nghĩa là gần nhau hơn về ngữ nghĩa; không phải xác suất chính xác.

In [11]:
query = "Tôi muốn tìm xe 7 chỗ rộng rãi cho gia đình đi du lịch"
query_embedding = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
scores = car_embeddings @ query_embedding

results = df.copy()
results["similarity"] = scores
results = results.sort_values("similarity", ascending=False).head(3)

print("Câu tìm kiếm:", query)
print("Kích thước vector câu hỏi:", query_embedding.shape)
display(results.reset_index(drop=True))

Câu tìm kiếm: Tôi muốn tìm xe 7 chỗ rộng rãi cho gia đình đi du lịch
Kích thước vector câu hỏi: (384,)


,name,description,similarity
0,Toyota Innova,"MPV 7 chỗ rộng rãi, phù hợp gia đình đông ngườ...",0.636978
1,Ford Everest,"SUV 7 chỗ gầm cao, mạnh mẽ, phù hợp đi đường d...",0.628631
2,Hyundai Creta,"SUV đô thị 5 chỗ, gầm cao, tiện nghi, phù hợp ...",0.575219


## 6. Thử tìm kiếm nhiều lần
Đổi nội dung `query` rồi chạy lại cell này. Không cần encode lại danh sách xe.

In [12]:
def search_cars(query, top_k=3):
    query_vector = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
    similarity = car_embeddings @ query_vector
    ranking = np.argsort(similarity)[::-1][:top_k]
    return pd.DataFrame([
        {"Xe": cars[i]["name"], "Mô tả": cars[i]["description"], "Cosine Similarity": round(float(similarity[i]), 4)}
        for i in ranking
    ])

query = "Tôi cần xe nhỏ gọn tiết kiệm xăng để đi làm"
display(search_cars(query))

,Xe,Mô tả,Cosine Similarity
0,Toyota Vios,"Sedan 5 chỗ nhỏ gọn, tiết kiệm nhiên liệu, phù...",0.7229
1,Honda City,"Sedan 5 chỗ tiết kiệm xăng, dễ lái, thích hợp ...",0.6918
2,Hyundai Creta,"SUV đô thị 5 chỗ, gầm cao, tiện nghi, phù hợp ...",0.5069
